In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
print(sys.path)

['/home/thule/miniconda3/envs/open-clip-training/lib/python311.zip', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/lib-dynload', '', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/site-packages', '../']


In [2]:
# from networks.resnet_big import ConResNet
# from torchinfo import summary
from thop import profile
import torch
import torch.nn as nn 

In [3]:
import torch
import torch.nn as nn
from thop import profile, clever_format
 
 
def count_macs_and_params(model: nn.Module, input_shape: tuple, device: str = "cpu"):
    model = model.to(device).eval()
    dummy_input = torch.randn(1, *input_shape).to(device)
 
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    flops = macs * 2  # quy đổi MACs -> FLOPs (1 MAC = 2 FLOPs)
    macs_str, params_str, flops_str = clever_format([macs, params, flops], "%.3f")
 
    return {
        "macs": int(macs),
        "flops": int(flops),
        "params": int(params),
        "macs_readable": macs_str,
        "flops_readable": flops_str,
        "params_readable": params_str,
    }

In [24]:
import time
import torch

def process_frame(model, frame_data, device):
    start_time = time.time()
    
    model.eval()
    frame_data = frame_data.to(device)
    
    with torch.no_grad(): 
        model(frame_data)
        
    end_time = time.time()
    processing_time = (end_time - start_time) * 1000
    return processing_time


In [88]:
from networks.cct_retnet import CCTRetNet
cct_retnet_model = CCTRetNet(
        img_size = 32,
        embedding_dim = 128, # the number of filters in the convolutional layers = the dimension of tokens before put into transformer encoder
        n_conv_layers = 1, # number of convolutional layers
        kernel_size = 3, # width = 8
        stride = 1, # width = 8
        padding = 1, # width = 0
        pooling_kernel_size = 3, # width = 1 
        pooling_stride = 2, # width = 1
        pooling_padding = 0,
        # num_layers = 4, # the number of transformer encoder layers
        # num_heads = 4, # the number of attention heads in each transformer encoder layer
        mlp_ratio = 2.5, # dim feedforward layer = embedding_dim * mlp_ratio, current case = 128 * 2.5 = 320
        num_classes = 10,
        positional_embedding = 'learnable', # ['sine', 'learnable', 'none']
        projection_dim=64,
        n_layers=2,
        n_heads=2
    )


In [89]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cct_retnet_model = cct_retnet_model.to(device)

In [29]:
print(count_macs_and_params(cct_retnet_model, input_shape=(2, 32, 32), device='cuda'))

{'macs': 28483072, 'flops': 56966144, 'params': 506817, 'macs_readable': '28.483M', 'flops_readable': '56.966M', 'params_readable': '506.817K'}


In [115]:
frame_data = torch.rand(1, 2, 32, 32)

processing_time = process_frame(cct_retnet_model, frame_data, device)

print(f"Thời gian xử lý một frame: {processing_time:.1f} ms")

Thời gian xử lý một frame: 16.8 ms


In [90]:
import time
import numpy as np
import torch


def benchmark_inference(model, frame_data, device, n_warmup=10, n_runs=100):
    model = model.to(device).eval()
    frame_data = frame_data.to(device)

    is_cuda = (device == "cuda" or (isinstance(device, torch.device) and device.type == "cuda"))

    # ── Bước 1: WARMUP - chạy vài lần đầu KHÔNG TÍNH ──
    with torch.no_grad():
        for _ in range(n_warmup):
            model(frame_data)
    if is_cuda:
        torch.cuda.synchronize()  # đợi warmup chạy XONG THẬT trước khi bắt đầu đo

    # ── Bước 2: ĐO NHIỀU LẦN, đồng bộ hóa GPU trước/sau MỖI lần ──
    times_ms = []
    with torch.no_grad():
        for _ in range(n_runs):
            if is_cuda:
                torch.cuda.synchronize()  # đảm bảo GPU đã rảnh TRƯỚC khi bắt đầu đếm giờ

            start_time = time.perf_counter()  # chính xác hơn time.time() cho benchmark
            model(frame_data)
            if is_cuda:
                torch.cuda.synchronize()  # đợi GPU tính XONG THẬT rồi mới dừng đồng hồ
            end_time = time.perf_counter()

            times_ms.append((end_time - start_time) * 1000)

    times_ms = np.array(times_ms)
    return {
        "mean_ms": float(times_ms.mean()),
        "std_ms": float(times_ms.std()),
        "min_ms": float(times_ms.min()),
        "max_ms": float(times_ms.max()),
        "all_times_ms": times_ms.tolist(),
    }

In [116]:
model = cct_retnet_model
frame_data = torch.rand(1, 2, 32, 32)

result = benchmark_inference(model, frame_data, device, n_warmup=10, n_runs=100)
print(f"Device: {device}")
print(f"Mean processing time: {result['mean_ms']:.3f} ms (± {result['std_ms']:.3f} ms)")
print(f"Min/Max: {result['min_ms']:.3f} ms / {result['max_ms']:.3f} ms")

Device: cuda
Mean processing time: 10.058 ms (± 3.526 ms)
Min/Max: 4.268 ms / 19.395 ms


In [117]:
model = cct_retnet_model
frame_data = torch.rand(1, 2, 32, 32)

result = benchmark_inference(model, frame_data, device, n_warmup=10, n_runs=100)
print(f"Device: {device}")
print(f"Mean processing time: {result['mean_ms']:.3f} ms (± {result['std_ms']:.3f} ms)")
print(f"Min/Max: {result['min_ms']:.3f} ms / {result['max_ms']:.3f} ms")

Device: cuda
Mean processing time: 8.678 ms (± 3.085 ms)
Min/Max: 3.961 ms / 17.333 ms
